# 🏆 FINAL CHAMPION TRAINING — `M7` (`MiT-B0 2-Stream + Focused+Rotate90 + BalancedProto-256`) FULL `960,000` CROPS (NO KD)

> **Nhà vô địch tuyệt đối từ bảng Full-Factorial $2^3 = 8$ (`16k × bs=8`):**
> * **`M7`** đứng **#1/8 trên TẤT CẢ chỉ số**: `@0.50 = 8.38%` IoU (`45.43%` Prec, `15.47%` F1) và `Peak = 12.44%` IoU (`22.12%` F1, `55.45%` mIoU) — vượt cả `Mask2Former (CVPR 2022)` chỉ bằng 1/7.5 ngân sách!
> * **Tối ưu On-Device Full-Scan:** `15.17M` params, `~70–72 GFLOPs` (**giảm 3.2× FLOPs so với M3 / 5.3× so với MiT-B2**), chạy Full-Scan 8 tiles chỉ ~**3.6 giây** trên CPU.
> * **Ngân sách Final chuẩn 100% Paper ASC-Former (Pattern Recognition 2025, Table 4):**
>   * **`80,000 iters × batch_size 12 = 960,000` crops** (~165.5 epochs), `AdamW lr=6e-5`, `Warmup 1,500 + PolyLR`, Native `BF16 + TF32`, **KHÔNG Knowledge Distillation** (thuần kiến trúc `M7`).
>   * **Ước tính thời gian chạy đơn trên Colab G4:** khoảng **~4.7 – 5.6 tiếng**.
>
> ### 🌙 HƯỚNG DẪN CHẠY NGẦM ĐI NGỦ (TẮT TRÌNH DUYỆT / TẮT MÁY AN TOÀN 100%):
> 1. Bấm **File → Save a copy in Drive** trước khi chạy *(bắt buộc cho Colab Pro+ / Ultra Background Execution!)*.
> 2. Chọn **Runtime → Change runtime type → G4 GPU**.
> 3. Chạy **Cell 1** (~3 phút) và bấm cấp quyền `drive.mount` khi cửa sổ bật lên.
> 4. Chạy **Cell 2 (All-in-One)**: đợi thấy dòng `Iter [200/80000]` chạy ổn định rồi **đóng trình duyệt & tắt máy đi ngủ thoải mái**!
>    * Cell 2 sẽ tự động: **Train 80,000 iters (backup Drive mỗi 10k + tự `--resume` nếu đứt) → Quét Full-Scan Sweep 1,203 ảnh cho cả `best_mIoU` & `iter_80000` → Xuất sẵn file ONNX INT8 lên Drive → Tự gọi `runtime.unassign()` TẮT GPU G4 để không đốt Compute Units lúc ngủ!**

### 🟢 CELL 1: KẾT NỐI DRIVE, CÀI MÔI TRƯỜNG & GIẢI NÉN DATASET (~3 Phút — Chạy có mặt để bấm cho phép Mount Drive)

In [ ]:
import os, sys, glob, torch
from google.colab import drive

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"🖥️ GPU: {props.name} | VRAM: {props.total_memory/(1024**3):.1f} GB | BF16: {torch.cuda.is_bf16_supported()}")
    print(f"   CPU cores: {os.cpu_count()} | PyTorch: {torch.__version__}")
else:
    raise RuntimeError("❌ Chưa bật GPU Runtime! Hãy chọn Runtime -> Change runtime type -> G4 GPU.")

drive.mount("/content/drive")

%cd /content
!rm -rf TMD
!git clone https://github.com/gagaduy/TMD.git

sys.path.insert(0, "/content/TMD/ASCFormer")
%env PYTHONPATH=/content/TMD/ASCFormer

!pip install -q yapf==0.40.1 mmcv-lite==2.0.0 mmengine==0.7.0 timm einops prettytable rich onnx onnxruntime gdown

GDRIVE_ZIP_URL = "https://drive.google.com/file/d/1uBtDzuTcqAFX9vWUH3n8Vn3EewWpW5kP/view?usp=drivesdk"

os.makedirs("/content/TMD/ASCFormer/data/ttd", exist_ok=True)
drive_zip = "/content/drive/MyDrive/RealTextMan.zip"
if not os.path.exists(drive_zip):
    matches = glob.glob("/content/drive/MyDrive/**/RealTextMan.zip", recursive=True)
    if matches:
        drive_zip = matches[0]
    elif GDRIVE_ZIP_URL.strip():
        drive_zip = "/content/RealTextMan.zip"
        print("🌐 Đang tải RealTextMan.zip từ link chia sẻ bằng gdown...")
        !gdown --fuzzy "{GDRIVE_ZIP_URL.strip()}" -O "{drive_zip}"
    else:
        raise FileNotFoundError("❌ Không tìm thấy RealTextMan.zip!")

print(f"📦 Đang giải nén {drive_zip} sang SSD Colab...")
!unzip -q -o "{drive_zip}" -d /content/TMD/ASCFormer/data/ttd/

img_cnt = len(glob.glob("/content/TMD/ASCFormer/data/ttd/RealTextMan/JPEGImages/*.jpg"))
print(f"✅ Dataset sẵn sàng: {img_cnt} ảnh JPEGImages! Bây giờ hãy chạy Cell 2 (All-in-One) rồi có thể tắt máy đi ngủ.")
assert img_cnt > 0

### 🌙 CELL 2 (ALL-IN-ONE): TRAIN `80,000 × bs=12` (`960k` CROPS) → FULL-SCAN SWEEP 1,203 ẢNH → EXPORT ONNX INT8 → TỰ TẮT GPU G4
> * Lưu toàn bộ kết quả vào `/content/drive/MyDrive/RTM_Final_B0_M7_BalProto_960k_G4/`.
> * Tự đồng bộ checkpoint mới mỗi `10,000` iters (`10k, 20k, ..., 80k` + `best_mIoU`) sang Drive và tự `--resume` nếu mất kết nối.
> * Để `AUTO_UNASSIGN_GPU = True` để Colab tự ngắt GPU G4 sau khi đã sao lưu xong toàn bộ lên Drive (tiết kiệm tối đa Compute Units!).

In [ ]:
import os, sys, glob, json, time, re, shutil, subprocess, torch
from prettytable import PrettyTable

# Bật True để tự tắt GPU G4 sau khi đã train + sweep + export + lưu Drive xong!
AUTO_UNASSIGN_GPU = True

%cd /content/TMD
!git pull origin main
%cd /content/TMD/ASCFormer
%env PYTHONPATH=/content/TMD/ASCFormer

CFG_PATH   = "configs/ascformer/ascformer_rtm_b0_m7_balproto_960k_g4.py"
DRIVE_DIR  = "/content/drive/MyDrive/RTM_Final_B0_M7_BalProto_960k_G4"
LOCAL_DIR  = "/content/work_dirs/final_b0_m7_balproto_960k"
os.makedirs(DRIVE_DIR, exist_ok=True)
os.makedirs(LOCAL_DIR, exist_ok=True)

# Tải trước Pretrained MiT-B0
_ = torch.hub.load_state_dict_from_url(
    "https://download.openmmlab.com/mmsegmentation/v0.5/pretrain/segformer/mit_b0_20220624-7e0fe6dd.pth",
    map_location="cpu"
)

def backup_to_drive(local_dir, drive_dir, synced_set=None):
    os.makedirs(drive_dir, exist_ok=True)
    for ext in ("*.pth", "*.log", "*.json", "*.py", "*.png", "*.txt", "*.md", "*.onnx"):
        for fp in glob.glob(os.path.join(local_dir, ext)):
            bn = os.path.basename(fp)
            if synced_set is not None and bn.endswith(".pth"):
                sig = (bn, os.path.getsize(fp), int(os.path.getmtime(fp)))
                if sig in synced_set:
                    continue
                shutil.copy2(fp, os.path.join(drive_dir, bn))
                synced_set.add(sig)
                print(f"   💾 [Drive Sync] Đã sao lưu {bn} ({os.path.getsize(fp)/1e6:.1f} MB)", flush=True)
            else:
                shutil.copy2(fp, os.path.join(drive_dir, bn))
    for fp in glob.glob(os.path.join(local_dir, "*", "*.log")) + glob.glob(os.path.join(local_dir, "*", "vis_data", "*.json")):
        shutil.copy2(fp, os.path.join(drive_dir, os.path.basename(fp)))
    for fp in glob.glob(os.path.join(local_dir, "**", "confusion_matrix.png"), recursive=True):
        shutil.copy2(fp, os.path.join(drive_dir, "confusion_matrix.png"))

def parse_progress(log_path):
    if not os.path.exists(log_path):
        return "Đang khởi tạo..."
    try:
        with open(log_path, "r", errors="ignore") as f:
            lines = f.readlines()[-50:]
        for ln in reversed(lines):
            if "Iter(train)" in ln:
                m = re.search(r"Iter\(train\)\s*\[\s*(\d+/\d+)\].*?eta:\s*([0-9:, ]+day[s]?,?\s*[0-9:]+|[0-9:]+).*?loss:\s*([0-9.]+).*?decode\.loss_ce:\s*([0-9.]+)", ln)
                if m:
                    return f"Iter [{m.group(1)}] | loss={m.group(3)} (ce={m.group(4)}) | ETA={m.group(2).strip()}"
            if "Iter(val)" in ln:
                return "Đang chạy Valid 1,203 ảnh..."
    except Exception:
        pass
    return "Đang nạp dữ liệu / compile..."

WORKER_TRAIN = "/tmp/run_final_m7_960k_train.py"
with open(WORKER_TRAIN, "w") as wf:
    wf.write("""import os, sys, runpy, torch
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True
_orig_load = torch.load
def _safe_load(*args, **kwargs):
    kwargs.setdefault('weights_only', False)
    return _orig_load(*args, **kwargs)
torch.load = _safe_load
try:
    import yapf.yapflib.yapf_api as _y
    import mmengine.config.config as _m
    _orig_fc = _y.FormatCode
    def _safe_fc(*args, **kwargs):
        kwargs.pop('verify', None)
        return _orig_fc(*args, **kwargs)
    _y.FormatCode = _safe_fc
    _m.FormatCode = _safe_fc
except Exception:
    pass
sys.argv = ['tools/train.py'] + sys.argv[1:]
runpy.run_path('tools/train.py', run_name='__main__')
""")

WORKER_SWEEP = "/tmp/run_final_m7_960k_sweep.py"
with open(WORKER_SWEEP, "w") as wf:
    wf.write("""import sys, runpy, torch
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
_orig_load = torch.load
def _safe_load(*args, **kwargs):
    kwargs.setdefault('weights_only', False)
    return _orig_load(*args, **kwargs)
torch.load = _safe_load
try:
    import yapf.yapflib.yapf_api as _y
    import mmengine.config.config as _m
    _orig_fc = _y.FormatCode
    def _safe_fc(*args, **kwargs):
        kwargs.pop('verify', None)
        return _orig_fc(*args, **kwargs)
    _y.FormatCode = _safe_fc
    _m.FormatCode = _safe_fc
except Exception:
    pass
sys.argv = ['tools/test_threshold_sweep.py'] + sys.argv[1:]
runpy.run_path('tools/test_threshold_sweep.py', run_name='__main__')
""")

t_global = time.time()
env = os.environ.copy()
env["PYTHONPATH"] = "/content/TMD/ASCFormer"
env["OMP_NUM_THREADS"] = "4"

try:
    # =========================================================================
    # GIAI ĐOẠN A: TRAIN 80,000 ITERS x BS=12 = 960,000 CROPS (AUTO-RESUME)
    # =========================================================================
    if os.path.exists(os.path.join(DRIVE_DIR, "iter_80000.pth")):
        print("✅ Đã có sẵn iter_80000.pth trên Drive — chuyển thẳng sang bước Sweep & Export!", flush=True)
    else:
        d_ckpts = sorted(glob.glob(os.path.join(DRIVE_DIR, "iter_*.pth")), key=os.path.getmtime)
        resume_args = []
        if d_ckpts:
            dst = os.path.join(LOCAL_DIR, os.path.basename(d_ckpts[-1]))
            shutil.copy2(d_ckpts[-1], dst)
            resume_args = ["--resume", dst]
            print(f"♻️ Tự động khôi phục (resume) từ: {os.path.basename(d_ckpts[-1])}", flush=True)

        log_path = os.path.join(LOCAL_DIR, "final_m7_960k_stdout.log")
        fh = open(log_path, "w")
        cmd = [sys.executable, WORKER_TRAIN, CFG_PATH, "--work-dir", LOCAL_DIR] + resume_args
        proc = subprocess.Popen(cmd, stdout=fh, stderr=subprocess.STDOUT, env=env)
        print(f"🚀 Bắt đầu train FINAL M7_BalancedProto_256 (80,000 x bs=12 = 960,000 crops) | PID={proc.pid}", flush=True)

        synced = set()
        while True:
            time.sleep(60)
            ret = proc.poll()
            elapsed_m = (time.time() - t_global) / 60.0
            gpu_stat = subprocess.check_output(
                ["nvidia-smi", "--query-gpu=memory.used,memory.total,utilization.gpu", "--format=csv,noheader"],
                text=True
            ).strip()
            if ret is None:
                print(f"⏱️ [{elapsed_m:5.1f}p] GPU ({gpu_stat}) | {parse_progress(log_path)}", flush=True)
                backup_to_drive(LOCAL_DIR, DRIVE_DIR, synced_set=synced)
            else:
                fh.close()
                if ret != 0:
                    backup_to_drive(LOCAL_DIR, DRIVE_DIR, synced_set=None)
                    print("❌ Train bị lỗi! 40 dòng log cuối:", flush=True)
                    with open(log_path, "r", errors="ignore") as lf:
                        print("".join(lf.readlines()[-40:]), flush=True)
                    raise RuntimeError(f"Training failed with exit code {ret}")
                print(f"🎉 HOÀN TẤT TRAIN 80,000 ITERS (960K CROPS) TRONG {elapsed_m:.1f} PHÚT! Đang sao lưu Drive...", flush=True)
                backup_to_drive(LOCAL_DIR, DRIVE_DIR, synced_set=None)
                break

    # =========================================================================
    # GIAI ĐOẠN B: FULL-SCAN SWEEP 1,203 ẢNH VAL CHO CẢ best_mIoU VÀ iter_80000
    # =========================================================================
    THRESHOLDS = [
        "0.05", "0.10", "0.15", "0.20", "0.25", "0.30",
        "0.35", "0.40", "0.45", "0.50", "0.55", "0.60", "0.65", "0.70"
    ]
    eval_targets = []
    best_cands = sorted(glob.glob(f"{LOCAL_DIR}/best_mIoU_*.pth") + glob.glob(f"{DRIVE_DIR}/best_mIoU_*.pth"), key=os.path.getmtime)
    last_cands = sorted(glob.glob(f"{LOCAL_DIR}/iter_80000.pth") + glob.glob(f"{DRIVE_DIR}/iter_80000.pth"), key=os.path.getmtime)
    if best_cands:
        eval_targets.append(("Best_mIoU_Ckpt", best_cands[-1]))
    if last_cands:
        eval_targets.append(("Iter_80000_Ckpt", last_cands[-1]))
    assert eval_targets, "❌ Không tìm thấy checkpoint để chạy Sweep!"

    summary = {}
    best_overall_ckpt = eval_targets[0][1]
    best_overall_iou = -1.0
    t_cmp = PrettyTable(["Checkpoint", "File", "@0.50 IoU", "@0.50 F1", "@0.50 Prec", "Peak tau*", "Peak IoU", "Peak F1", "Peak mIoU"])

    for tag, ckpt_p in eval_targets:
        out_json = os.path.join(DRIVE_DIR, f"M7_960k_{tag}_sweep.json")
        if not os.path.exists(out_json):
            print(f"\n🔍 Đang chạy Full-Scan Sweep 1,203 ảnh cho [{tag}] ({os.path.basename(ckpt_p)})...", flush=True)
            sweep_wd = os.path.join(LOCAL_DIR, f"sweep_{tag}")
            cmd_sw = [
                sys.executable, WORKER_SWEEP, CFG_PATH, ckpt_p,
                "--ann-file", "val.txt", "--work-dir", sweep_wd,
                "--thresholds", *THRESHOLDS, "--out-json", out_json
            ]
            subprocess.run(cmd_sw, env=env, check=True)
        res = json.load(open(out_json))["results"]
        r50 = [r for r in res if abs(r["threshold"] - 0.50) < 1e-4][0]
        rpk = max(res, key=lambda r: r["tampered_iou"])
        summary[tag] = {"file": os.path.basename(ckpt_p), "tau_050": r50, "peak": rpk}
        t_cmp.add_row([
            tag, os.path.basename(ckpt_p),
            f"{r50['tampered_iou']:.2f}%", f"{r50['tampered_dice']:.2f}%", f"{r50['tampered_precision']:.2f}%",
            f"{rpk['threshold']:.3f}", f"{rpk['tampered_iou']:.2f}%", f"{rpk['tampered_dice']:.2f}%", f"{rpk['m_iou']:.2f}%"
        ])
        if rpk["tampered_iou"] > best_overall_iou:
            best_overall_iou = rpk["tampered_iou"]
            best_overall_ckpt = ckpt_p

    sum_json = os.path.join(DRIVE_DIR, "FINAL_M7_960k_summary.json")
    rep_txt  = os.path.join(DRIVE_DIR, "FINAL_M7_960k_report.txt")
    rep_md   = os.path.join(DRIVE_DIR, "FINAL_M7_960k_report.md")
    with open(sum_json, "w") as f:
        json.dump(summary, f, indent=2)
    with open(rep_txt, "w") as f:
        f.write("KẾT QUẢ FINAL M7_BalancedProto_256 (80,000 iters x bs=12 = 960,000 crops, Full 1,203 ảnh val)\n")
        f.write(str(t_cmp) + "\n")
    with open(rep_md, "w") as f:
        f.write("# Kết quả Final MiT-B0 Champion (`M7_BalancedProto_256`, `960k` crops, KHÔNG KD)\n\n")
        f.write("| Checkpoint | File | @0.50 IoU | @0.50 F1 | @0.50 Prec | Peak tau* | Peak IoU | Peak F1 | Peak mIoU |\n")
        f.write("| :--- | :--- | :-: | :-: | :-: | :-: | :-: | :-: | :-: |\n")
        for tag, info in summary.items():
            r50, rpk = info["tau_050"], info["peak"]
            f.write(f"| {tag} | `{info['file']}` | {r50['tampered_iou']:.2f}% | {r50['tampered_dice']:.2f}% | {r50['tampered_precision']:.2f}% | {rpk['threshold']:.3f} | **{rpk['tampered_iou']:.2f}%** | **{rpk['tampered_dice']:.2f}%** | {rpk['m_iou']:.2f}% |\n")

    print("\n🏆 BẢNG KẾT QUẢ CHÍNH THỨC FINAL M7_BalancedProto_256 (960,000 CROPS, 1,203 ẢNH VAL):")
    print(t_cmp, flush=True)

    # =========================================================================
    # GIAI ĐOẠN C: TỰ ĐỘNG XUẤT ONNX INT8 SẴN SÀNG NẠP VÀO APP ANDROID
    # =========================================================================
    try:
        onnx_int8 = os.path.join(LOCAL_DIR, "rtm_b0_m7_balproto_960k_int8.onnx")
        print(f"\n📱 Đang tự động xuất ONNX INT8 từ checkpoint mạnh nhất ({os.path.basename(best_overall_ckpt)})...", flush=True)
        subprocess.run([
            sys.executable, "tools/export_ondevice_model.py",
            "--config", CFG_PATH,
            "--checkpoint", best_overall_ckpt,
            "--output", onnx_int8
        ], env=env, check=True)
        shutil.copy2(onnx_int8, os.path.join(DRIVE_DIR, os.path.basename(onnx_int8)))
        print(f"✅ Đã lưu sẵn ONNX INT8 lên Drive: {os.path.join(DRIVE_DIR, os.path.basename(onnx_int8))} ({os.path.getsize(onnx_int8)/1e6:.2f} MB)", flush=True)
    except Exception as ex:
        print(f"⚠️ Export ONNX gặp lỗi nhẹ ({ex}) — toàn bộ checkpoint & kết quả Sweep đã lưu an toàn trên Drive!", flush=True)

    print(f"\n✅ TOÀN BỘ QUY TRÌNH HOÀN TẤT TRONG {(time.time()-t_global)/60.0:.1f} PHÚT! Thư mục Drive: {DRIVE_DIR}", flush=True)

finally:
    # Flush toàn bộ ổ đĩa & Drive FUSE trước khi tự tắt GPU G4
    try:
        os.sync()
    except Exception:
        pass
    if AUTO_UNASSIGN_GPU:
        print("🌙 Đợi 20 giây cho Google Drive đồng bộ xong hoàn toàn rồi tự động tắt GPU G4 tiết kiệm Compute Units...", flush=True)
        time.sleep(20)
        try:
            from google.colab import runtime
            runtime.unassign()
        except Exception as e:
            print(f"⚠️ Không gọi được runtime.unassign(): {e}", flush=True)